In [1]:
import optuna
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('balanced.csv')
col =['Sex','MaritalStatus']
ct = ColumnTransformer([('oe',OneHotEncoder(),col)],remainder='passthrough')
x = df.drop('FraudFound_P',axis=1)
x = ct.fit_transform(x)
y = df['FraudFound_P']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True,stratify=y)

In [3]:
from sklearn.model_selection import cross_validate
from xgboost import XGBClassifier

def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 1),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1)
    }
    
    model = XGBClassifier(**param)
    
    cv_results = cross_validate(
        model, x_train, y_train, cv=5, 
        scoring=['accuracy', 'f1'], 
        n_jobs=-1
    )
    
    accuracy = cv_results['test_accuracy'].mean()
    f1 = cv_results['test_f1'].mean()
    
    return accuracy, f1




In [4]:
study = optuna.create_study(directions=["maximize", "maximize"], sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=150)


[I 2026-05-08 16:35:55,839] A new study created in memory with name: no-name-a95969c4-ffb0-4dfa-a67d-712799ba47a8
[I 2026-05-08 16:36:02,018] Trial 0 finished with values: [0.6527630285152408, 0.6656975428385274] and parameters: {'n_estimators': 850, 'max_depth': 6, 'learning_rate': 0.0742642607301075, 'subsample': 0.6339729972883185, 'colsample_bytree': 0.9285185964602691, 'min_child_weight': 9, 'gamma': 0.5828661156867854, 'reg_alpha': 0.15472809610392435, 'reg_lambda': 0.8174949803971586}.
[I 2026-05-08 16:36:05,206] Trial 1 finished with values: [0.7015889872173059, 0.7232104013064106] and parameters: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.02755429380110628, 'subsample': 0.5024203258106505, 'colsample_bytree': 0.554835329722021, 'min_child_weight': 2, 'gamma': 0.8582349908530723, 'reg_alpha': 0.8922053720345862, 'reg_lambda': 0.7005539266375088}.
[I 2026-05-08 16:36:05,517] Trial 2 finished with values: [0.6811799410029499, 0.6931752240102727] and parameters: {'n_

In [5]:
for i, trial in enumerate(study.best_trials):
    print(f"\nTrial {i}:")
    print(f"  Accuracy: {trial.values[0]:.4f}, F1: {trial.values[1]:.4f}")
    print(f"  Params: {trial.params}")



Trial 0:
  Accuracy: 0.7149, F1: 0.7392
  Params: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.026737883139194202, 'subsample': 0.5647500962386196, 'colsample_bytree': 0.5409876988102682, 'min_child_weight': 9, 'gamma': 0.9228014407385188, 'reg_alpha': 0.9310839385345714, 'reg_lambda': 0.9827760741073778}


In [6]:
def objective(trial):
    
    param = {
        'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf']),
        'C': trial.suggest_float('C', 1e-3, 100, log=True),
        'gamma': trial.suggest_float('gamma', 1e-3, 100, log=True),
        'probability': True 
    }
    model = SVC(**param)
    
    cv_results = cross_validate(
        model, x_train, y_train, cv=5, 
        scoring=['accuracy', 'f1'], 
        n_jobs=-1
    )
    
    accuracy = cv_results['test_accuracy'].mean()
    f1 = cv_results['test_f1'].mean()
    
    return accuracy, f1

In [7]:
study = optuna.create_study(directions=["maximize", "maximize"], sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

[I 2026-05-08 16:36:27,947] A new study created in memory with name: no-name-934c9842-0998-46d9-8b21-fbed0cf4571e
[I 2026-05-08 16:36:28,412] Trial 0 finished with values: [0.5114965585054081, 0.3844702252289079] and parameters: {'kernel': 'rbf', 'C': 0.11277165895887031, 'gamma': 0.29013845449352627}.
[I 2026-05-08 16:36:28,781] Trial 1 finished with values: [0.7069262536873157, 0.7322462318633486] and parameters: {'kernel': 'rbf', 'C': 51.120592320806686, 'gamma': 0.0010530360244200804}.
[I 2026-05-08 16:36:32,772] Trial 2 finished with values: [0.7113588987217305, 0.7324473158391231] and parameters: {'kernel': 'linear', 'C': 17.976325937742708, 'gamma': 4.368186805351205}.
[I 2026-05-08 16:36:33,149] Trial 3 finished with values: [0.5026470009832842, 0.3458753709198813] and parameters: {'kernel': 'rbf', 'C': 0.016939434600952957, 'gamma': 2.9142065645303665}.
[I 2026-05-08 16:36:33,588] Trial 4 finished with values: [0.46973844641101276, 0.33827946816264076] and parameters: {'kernel

In [8]:
for i, trial in enumerate(study.best_trials):
    print(f"\nTrial {i}:")
    print(f"  Accuracy: {trial.values[0]:.4f}, F1: {trial.values[1]:.4f}")
    print(f"  Params: {trial.params}")



Trial 0:
  Accuracy: 0.7105, F1: 0.7414
  Params: {'kernel': 'linear', 'C': 0.00687024587033703, 'gamma': 2.280645001638757}

Trial 1:
  Accuracy: 0.7114, F1: 0.7329
  Params: {'kernel': 'linear', 'C': 0.08324577136483163, 'gamma': 1.1886487315172336}

Trial 2:
  Accuracy: 0.7114, F1: 0.7329
  Params: {'kernel': 'linear', 'C': 0.11913918547032208, 'gamma': 1.213755629674641}

Trial 3:
  Accuracy: 0.7114, F1: 0.7329
  Params: {'kernel': 'linear', 'C': 0.6109101110757854, 'gamma': 23.519056698736748}

Trial 4:
  Accuracy: 0.7114, F1: 0.7329
  Params: {'kernel': 'linear', 'C': 0.26597219821024565, 'gamma': 0.13720810873970082}

Trial 5:
  Accuracy: 0.7114, F1: 0.7329
  Params: {'kernel': 'linear', 'C': 0.054111044176155566, 'gamma': 0.9265460290932128}

Trial 6:
  Accuracy: 0.7078, F1: 0.7492
  Params: {'kernel': 'linear', 'C': 0.012398156691089344, 'gamma': 25.698553468915993}

Trial 7:
  Accuracy: 0.7114, F1: 0.7329
  Params: {'kernel': 'linear', 'C': 0.6032235372751888, 'gamma': 27.17

In [9]:
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000,step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', None])
    }
    model = RandomForestClassifier(**param)
    cv_results = cross_validate(
        model, x_train, y_train, cv=5, 
        scoring=['accuracy', 'f1'], 
        n_jobs=-1
    )
    
    accuracy = cv_results['test_accuracy'].mean()
    f1 = cv_results['test_f1'].mean()
    
    return accuracy, f1

In [10]:
study = optuna.create_study(directions=["maximize", "maximize"], sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=150)

[I 2026-05-08 16:37:06,320] A new study created in memory with name: no-name-f69d1fd7-5978-4af2-b1a1-5194b65fac5f
[I 2026-05-08 16:37:06,890] Trial 0 finished with values: [0.6909459193706982, 0.710658892145408] and parameters: {'n_estimators': 200, 'max_depth': 9, 'min_samples_split': 9, 'min_samples_leaf': 1, 'criterion': 'entropy', 'class_weight': 'balanced'}.
[I 2026-05-08 16:37:07,920] Trial 1 finished with values: [0.7095811209439529, 0.7276856291563462] and parameters: {'n_estimators': 450, 'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 6, 'criterion': 'entropy', 'class_weight': None}.
[I 2026-05-08 16:37:09,455] Trial 2 finished with values: [0.7095811209439528, 0.7308487763692806] and parameters: {'n_estimators': 600, 'max_depth': 3, 'min_samples_split': 8, 'min_samples_leaf': 4, 'criterion': 'gini', 'class_weight': 'balanced'}.
[I 2026-05-08 16:37:10,165] Trial 3 finished with values: [0.7104700098328417, 0.7287495057635572] and parameters: {'n_estimators': 350, 

In [11]:
for i, trial in enumerate(study.best_trials):
    print(f"\nTrial {i}:")
    print(f"  Accuracy: {trial.values[0]:.4f}, F1: {trial.values[1]:.4f}")
    print(f"  Params: {trial.params}")



Trial 0:
  Accuracy: 0.7122, F1: 0.7335
  Params: {'n_estimators': 550, 'max_depth': 3, 'min_samples_split': 5, 'min_samples_leaf': 2, 'criterion': 'gini', 'class_weight': 'balanced'}

Trial 1:
  Accuracy: 0.7131, F1: 0.7310
  Params: {'n_estimators': 400, 'max_depth': 4, 'min_samples_split': 7, 'min_samples_leaf': 3, 'criterion': 'entropy', 'class_weight': 'balanced'}

Trial 2:
  Accuracy: 0.7122, F1: 0.7335
  Params: {'n_estimators': 200, 'max_depth': 3, 'min_samples_split': 6, 'min_samples_leaf': 2, 'criterion': 'entropy', 'class_weight': 'balanced'}

Trial 3:
  Accuracy: 0.7122, F1: 0.7335
  Params: {'n_estimators': 450, 'max_depth': 3, 'min_samples_split': 6, 'min_samples_leaf': 3, 'criterion': 'entropy', 'class_weight': 'balanced'}


In [12]:
# nb

def objective(trial):
    param = {
        'var_smoothing': trial.suggest_float('var_smoothing', 1e-3, 1e-1, log=True)
    }
    model = GaussianNB()
    cv_results = cross_validate(
        model, x_train, y_train, cv=5, 
        scoring=['accuracy', 'f1'], 
        n_jobs=-1
    )
    
    accuracy = cv_results['test_accuracy'].mean()
    f1 = cv_results['test_f1'].mean()
    
    return accuracy, f1

In [13]:
study = optuna.create_study(directions=["maximize", "maximize"], sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=150)

[I 2026-05-08 16:39:44,992] A new study created in memory with name: no-name-6ac373eb-b71c-4ad6-9a23-45abf590db7f
[I 2026-05-08 16:39:45,038] Trial 0 finished with values: [0.7122477876106196, 0.7324362198325652] and parameters: {'var_smoothing': 0.0034447309185349113}.
[I 2026-05-08 16:39:45,064] Trial 1 finished with values: [0.7122477876106196, 0.7324362198325652] and parameters: {'var_smoothing': 0.022650923852859793}.
[I 2026-05-08 16:39:45,092] Trial 2 finished with values: [0.7122477876106196, 0.7324362198325652] and parameters: {'var_smoothing': 0.08050585885209278}.
[I 2026-05-08 16:39:45,119] Trial 3 finished with values: [0.7122477876106196, 0.7324362198325652] and parameters: {'var_smoothing': 0.013095551453235573}.
[I 2026-05-08 16:39:45,144] Trial 4 finished with values: [0.7122477876106196, 0.7324362198325652] and parameters: {'var_smoothing': 0.0036395289232569387}.
[I 2026-05-08 16:39:45,169] Trial 5 finished with values: [0.7122477876106196, 0.7324362198325652] and pa

In [14]:
for i, trial in enumerate(study.best_trials):
    print(f"\nTrial {i}:")
    print(f"  Accuracy: {trial.values[0]:.4f}, F1: {trial.values[1]:.4f}")
    print(f"  Params: {trial.params}")



Trial 0:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.0034447309185349113}

Trial 1:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.022650923852859793}

Trial 2:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.08050585885209278}

Trial 3:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.013095551453235573}

Trial 4:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.0036395289232569387}

Trial 5:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.017159374258377186}

Trial 6:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.006116212285786978}

Trial 7:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.0018297460318147422}

Trial 8:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.0024248660651711345}

Trial 9:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.05861219290061891}

Trial 10:
  Accuracy: 0.7122, F1: 0.7324
  Params: {'var_smoothing': 0.0011431637275125

In [ ]:
#nb Accuracy: 0.7122, F1: 0.7324
#rf Accuracy: 0.7122, F1: 0.7335
# svc Accuracy: 0.7105, F1: 0.7414
# xgb Accuracy: 0.7149, F1: 0.7392 best
